# Protein domain correlation contrast, from the master table

Master-table version of [`protein_domains_correlations.ipynb`](protein_domains_correlations.ipynb):
same comparison — per (gene, trait, annotation) Spearman correlation split by whether the
variant falls inside a protein domain/interface region vs. outside it — but reads **only**
`MASTER_PATH` in place of the separate annotation, association, correlation and genebass-beta
files the parent notebook joins by hand.

Shared setup (§0) is copied verbatim from
[`../master_file_correlations.ipynb`](../master_file_correlations.ipynb) so both notebooks stay
interchangeable; §1 adds the domain-flag derivation and the in-domain/not-in-domain contrast.

Master table built by [`utils/create_master_table.ipynb`](../utils/create_master_table.ipynb).
Grain is one row per `(id, region)`; gene/trait columns (`phenotype`, `loftee_corr_dir`, …)
are already attached to every variant row.

## 0. Shared setup

(identical to `../master_file_correlations.ipynb` §0)

In [ ]:
import yaml
import numpy as np
import polars as pl

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import env_override, fetch_hf_data
MASTER_PATH = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
CONFIG_DIR  = str(REPO_ROOT / 'configs')
FIG_DIR     = env_override('FIG_DIR', str(REPO_ROOT / 'paper_figures'))

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import *

_THEME = theme_minimal() + theme(
    axis_text=element_text(size=11, lineheight=1.4),
    axis_title=element_text(size=12),
    legend_text=element_text(size=12),
    legend_title=element_text(size=12),
    plot_background=element_rect(fill='white', color='white'),
)

MASTER_COLS = pl.scan_parquet(MASTER_PATH).collect_schema().names()
DERIVED_COLS = derived_schema(MASTER_PATH)
print(f'master table: {len(MASTER_COLS)} cols -> {len(DERIVED_COLS)} after add_derived()')


## 1. In-domain vs. not-in-domain correlation contrast

Per (gene, trait, annotation) Spearman correlation between the annotation score and the
genebass beta, split by a domain/interface flag, direction-corrected. Reproduces
`protein_domains_correlations.ipynb`.

`D1_domain_type` picks which boolean split to contrast — `ted_domain` (TED domain calls),
`plddt_structured` (AlphaFold pLDDT ≥ 70), or `interface_all` (any of: pioneer high-confidence
interface, PDB inter-chain non-bonded contact, PDB inter-chain H-bond, genomics2proteins
interface). All three are computed directly from master-table columns — the master table's own
`add_derived()` above already keeps `high_plddt`/`low_plddt` (>70 threshold) for a different
purpose, so `plddt_structured` here uses its own ≥70 threshold to match the parent notebook
exactly.

In [ ]:
# --- parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override without editing)
D1_variant_class       = env_override('VARIANT_CLASS', 'missense')
D1_config               = env_override('CONFIG_FILE', 'config_correlations.yaml')
D1_selected_categories = env_override('SELECTED_CATEGORIES',
                                       ['missense', 'genetic_diversity', 'gnomad', 'conservation'], 'list')
D1_mac                 = env_override('MAC', 20, int)
D1_only_snps           = env_override('ONLY_SNPS', True, bool)
D1_only_clinvar        = env_override('ONLY_CLINVAR', False, bool)
D1_exclude_clinvar     = env_override('EXCLUDE_CLINVAR', False, bool)
D1_min_variants        = env_override('MIN_VARIANTS', 50, int)      # region-phenotype-domain groups below this are dropped
D1_domain_type         = env_override('DOMAIN_TYPE', 'plddt_structured')   # 'ted_domain', 'plddt_structured', or 'interface_all'
D1_ci_factor           = env_override('CI_FACTOR', 1.96, float)
D1_paper_tools         = ["CPT-1", "BayesDel", "REVEL", "ClinPred", "AlphaMissense",
                           "CADD Raw", "PolyPhen2", "ESM1v", "GPN-MSA", "Vertebrate PhyloP"]

DOMAIN_EXPRS = {
    'ted_domain': pl.col('ted_domain') == True,
    'plddt_structured': pl.col('plddt') >= 70,
    'interface_all': pl.any_horizontal(
        pl.col('is_pioneer_interface_high') == True,
        pl.col('inter_chain_non_bonded_interaction_pdb_count') > 0,
        pl.col('inter_chain_hydrogen_bond_pdb_count') > 0,
        pl.col('interface_genomics2proteins') == True,
    ),
}

In [ ]:
d1_cfg, d1_all = load_config(CONFIG_DIR, D1_config)
d1_vc  = load_variant_class(CONFIG_DIR, D1_variant_class)
d1_lf  = (scan_variants(MASTER_PATH, d1_vc, only_snps=D1_only_snps,
                        only_clinvar=D1_only_clinvar, exclude_clinvar=D1_exclude_clinvar)
          .with_columns(in_domain=DOMAIN_EXPRS[D1_domain_type]))
d1_annos = pick_annos(d1_cfg, d1_all, D1_selected_categories, DERIVED_COLS)


domain_correlation_df = gene_trait_tool_correlations(
    d1_lf, D1_mac, d1_annos, d1_cfg, D1_selected_categories, D1_min_variants,
    extra_group_cols=('in_domain',),
)
print(domain_correlation_df.shape)
domain_correlation_df.head()

In [ ]:
# domain_correlation_df already has the coverage + min-variants filter applied (previous cell)
d1_filt = domain_correlation_df.drop('n_variants')

# Only keep annotation-region-phenotype triples scored on both sides of the split
d1_paired = (d1_filt.group_by(['annotation', 'region', 'phenotype'])
    .agg(n_domains=pl.col('in_domain').n_unique())
    .filter(pl.col('n_domains') == 2)
    .select(['annotation', 'region', 'phenotype']))

domain_corr_df = (
    d1_filt.join(d1_paired, on=['annotation', 'region', 'phenotype'], how='semi')
    .pivot(on='in_domain', values='corr_beta',
           index=['region', 'phenotype', 'annotation', 'label', 'color', 'category'])
    .drop_nulls()
    .rename({'true': 'domain_corr', 'false': 'not_domain_corr'})
)
print(domain_corr_df.shape)
domain_corr_df.head()

In [ ]:
d1_summary = (
    domain_corr_df
    .group_by(['annotation', 'label', 'color', 'category'])
    .agg(
        n_assocs=pl.len(),
        mean_domain=pl.col('domain_corr').mean(),
        std_domain=pl.col('domain_corr').std(),
        mean_not_domain=pl.col('not_domain_corr').mean(),
        std_not_domain=pl.col('not_domain_corr').std(),
    )
    .with_columns(
        sem_domain=pl.col('std_domain') / pl.col('n_assocs').sqrt(),
        sem_not_domain=pl.col('std_not_domain') / pl.col('n_assocs').sqrt(),
        mean_diff=pl.col('mean_domain') - pl.col('mean_not_domain'),
        sem_diff=((pl.col('std_domain') ** 2 + pl.col('std_not_domain') ** 2) / pl.col('n_assocs')).sqrt(),
    )
    .with_columns(
        domain_ci_lower=pl.col('mean_domain') - D1_ci_factor * pl.col('sem_domain'),
        domain_ci_upper=pl.col('mean_domain') + D1_ci_factor * pl.col('sem_domain'),
        not_domain_ci_lower=pl.col('mean_not_domain') - D1_ci_factor * pl.col('sem_not_domain'),
        not_domain_ci_upper=pl.col('mean_not_domain') + D1_ci_factor * pl.col('sem_not_domain'),
        diff_ci_lower=pl.col('mean_diff') - D1_ci_factor * pl.col('sem_diff'),
        diff_ci_upper=pl.col('mean_diff') + D1_ci_factor * pl.col('sem_diff'),
    )
)
d1_cat_color = dict(
    d1_cfg.filter(pl.col('annotation').is_in(d1_summary['annotation']))
    .group_by('category').agg(pl.col('color').first()).iter_rows()
)
d1_summary

In [ ]:
d1_scatter = (
    ggplot(d1_summary, aes(x='mean_domain', y='mean_not_domain'))
    # + geom_abline(intercept=0, slope=1, linetype='dashed', color='grey')
    # + geom_errorbar(aes(ymin='not_domain_ci_lower', ymax='not_domain_ci_upper'), width=0, color='black')
    # + geom_errorbarh(aes(xmin='domain_ci_lower', xmax='domain_ci_upper'), height=0, color='black')
    + geom_point(aes(fill='category'), color='black', size=4, stroke=0.5)
    + geom_text(aes(label='label'), color='black', size=13, va='bottom', nudge_y=0.0005)
    + scale_fill_manual(values=d1_cat_color, name='Tool category')
    + labs(x=f'Mean Spearman ρ in {D1_domain_type}', y=f'Mean Spearman ρ outside {D1_domain_type}')
    + _THEME 
    + theme(
        figure_size=(8, 7),
        legend_position='bottom',
        axis_title=element_text(size=14),
        axis_text=element_text(size=12),
    )
)
d1_scatter.save(f'{FIG_DIR}/FS8_domain_corr_contrast_{D1_variant_class}_{D1_domain_type}.svg', dpi=200, verbose=False)
d1_scatter

In [ ]:
# Forest plot of the domain - not-domain mean correlation difference, all tools
d1_order = d1_summary.sort('mean_diff')['label']
d1_diff_df = d1_summary.with_columns(pl.col('label').cast(pl.Enum(d1_order)))

d1_diff_plot = (
    ggplot(d1_diff_df, aes(x='label', y='mean_diff'))
    + geom_hline(yintercept=0, linetype='dotted', color='grey')
    + geom_errorbar(aes(ymin='diff_ci_lower', ymax='diff_ci_upper'), width=0.0001, color='black')
    + geom_point(aes(fill='category'), color='black', size=4, stroke=0.5)
    + scale_fill_manual(values=d1_cat_color, name='Tool category')
    + coord_flip()
    + labs(x='', y='Mean Spearman ρ difference ± 1.96×SEM\n(direction-corrected)')
    + _THEME 
    + theme(
        figure_size=(7, d1_diff_df.height * 0.35 + 1),
        legend_position='bottom',
        axis_title=element_text(size=14),
        axis_text=element_text(size=13),
    )
)
d1_diff_plot.save(f'{FIG_DIR}/FS8_domain_corr_diff_{D1_variant_class}_{D1_domain_type}.svg', dpi=200, verbose=False)
d1_diff_plot